In [ ]:
import torch
import torch.nn as nn
import math
import os
from tokenizers import ByteLevelBPETokenizer
from huggingface_hub import hf_hub_download, snapshot_download


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Χρησιμοποιείται συσκευή: {device}")


class ERPLanguageModel(nn.Module):
    def __init__(self, vocab_size=8192, d_model=512, n_heads=8, n_layers=8, max_seq_len=256):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True,
            activation="gelu",
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0)
        x = self.token_embedding(x) * math.sqrt(self.d_model) + self.position_embedding(positions)
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
        x = self.transformer(x, mask=mask, is_causal=True)
        logits = self.lm_head(x)
        return logits


hf_repo_id = "deanwashburn/Agro_30M"


print("Λήψη αρχείων tokenizer από Hugging Face...")

repo_local_path = snapshot_download(repo_id=hf_repo_id)

tokenizer = ByteLevelBPETokenizer(
    vocab=os.path.join(repo_local_path, "vocab.json"),
    merges=os.path.join(repo_local_path, "merges.txt")
)
print("Ο tokenizer φορτώθηκε επιτυχώς!")


print("Λήψη βαρών μοντέλου από Hugging Face...")

checkpoint_path = hf_hub_download(repo_id=hf_repo_id, filename="erp_transformer_30m.pt")
checkpoint = torch.load(checkpoint_path, map_location=device)


hp = checkpoint['hyperparameters']
model = ERPLanguageModel(
    vocab_size=hp['vocab_size'],
    d_model=hp['d_model'],
    n_heads=hp['n_heads'],
    n_layers=hp['n_layers'],
    max_seq_len=hp['max_seq_len']
).to(device)


model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("Το μοντέλο φορτώθηκε επιτυχώς και είναι έτοιμο!")

# 5. Συνάρτηση Παραγωγής Κειμένου (Inference Loop)
@torch.no_grad()
def generate_text(prompt_text, max_new_tokens=30, temperature=0.8):
    model.eval()
    encoded = tokenizer.encode(prompt_text)
    x = torch.tensor([encoded.ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        x_cond = x[:, -model.max_seq_len:]
        logits = model(x_cond)
        next_token_logits = logits[:, -1, :] / temperature
        probs = torch.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_token], dim=1)

    return tokenizer.decode(x[0].tolist())

# 6. Δοκιμή παραγωγής με ένα ERP prompt
prompt = "Παραγγελία"
output = generate_text(prompt, max_new_tokens=25, temperature=0.7)

print("-" * 40)
print(f"Prompt: {prompt}")
print(f"Generated Output: {output}")
print("-" * 40)

Χρησιμοποιείται συσκευή: cuda
Λήψη αρχείων tokenizer από Hugging Face...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Ο tokenizer φορτώθηκε επιτυχώς!
Λήψη βαρών μοντέλου από Hugging Face...


/tmp/ipykernel_4544/1214274799.py:32: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


Το μοντέλο φορτώθηκε επιτυχώς και είναι έτοιμο!
----------------------------------------
Prompt: Παραγγελία
Generated Output: Παραγγελία #9012: 50 τεμ antallaktika gia trakter Massey Ferguson, τιμή location.50 EUR.50 EUR.50 EUR.
----------------------------------------
